# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanaahmedradwan123-commits/flyrank-internship-w1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

Freestyle. **Question: which pages are already earning traffic from AI answer engines
(ChatGPT, Perplexity, AI Overviews), and what distinguishes them from pages that aren't?**

`ai_sessions_90d` / `ai_traffic_pct` show up in the data dictionary but none of the four
predefined lanes are built around them — and the pattern looked worth chasing the moment I
saw how rare and unevenly distributed it is (see numbers below). This also maps to a real,
currently-unsolved decision: FlyRank's content team doesn't yet have a systematic way to say
"this page is worth restructuring for AI-citation" versus "leave it as a normal organic page."
That's a live gap a small, honest model can help close, even if the label available to me
right now (`ai_traffic_pct > 0`) is a rough, anonymized proxy rather than ground truth on
actual AI-engine citations.


In [2]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")
print("rows, cols:", df.shape)
print("clients:", df["client_id"].nunique())
print("pages with any AI-assistant referral traffic:", (df["ai_traffic_pct"] > 0).sum(),
      f"({(df['ai_traffic_pct'] > 0).mean()*100:.1f}% of pages)")


rows, cols: (9349, 44)
clients: 32
pages with any AI-assistant referral traffic: 568 (6.1% of pages)


## 2. The question: decision, action, cost of a wrong call

- **Unit of analysis:** a page (`content_id`).
- **Output:** a rank-ordered list (or binary flag) of pages that are strong candidates for
  "AI-visibility" editorial work — restructuring for direct-answer / extractable formatting.
- **Decision it supports:** which pages a content editor should prioritize for AI-visibility
  restructuring in the next work cycle, out of the much larger pile of pages that could
  theoretically be touched.
- **Action a human takes:** an editor pulls the top-N flagged pages and rewrites/restructures
  them (clearer direct answers, better structured data, tighter Q&A framing) instead of
  guessing which pages to start with.
- **Cost of a wrong call:** a false positive burns editorial hours restructuring a page that
  was never going to pick up AI-referral traffic (pure opportunity cost); a false negative
  leaves a page that could have gained a new, currently-growing acquisition channel sitting
  untouched, so the client keeps missing out on it.
- **Why data/ML helps at all:** only ~6% of pages in this sample show any AI-referral
  traffic at all, and no single column I checked (position, word count, freshness, age)
  correlates with it in isolation — the signal, if it exists, is a combination of features,
  which is exactly the kind of pattern a human skimming a spreadsheet is bad at spotting and
  a simple evaluated model is well-suited to surface.

In [3]:
# Sanity check backing the decision framing above: confirm this is genuinely
# spread across many separate clients/accounts, not one outlier skewing everything.
print("distinct clients in the sample:", df["client_id"].nunique())


distinct clients in the sample: 32


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [4]:
has_ai = df["ai_traffic_pct"] > 0

print(f"1) Base rate: {has_ai.mean()*100:.2f}% of pages ({has_ai.sum()} of {len(df)}) show any AI-assistant referral traffic.")

print(f"2) Those pages already run hotter on organic signals: median clicks_90d is "
      f"{df.loc[has_ai, 'clicks_90d'].median():.0f} vs {df.loc[~has_ai, 'clicks_90d'].median():.0f} for the rest; "
      f"mean engagement_rate is {df.loc[has_ai, 'engagement_rate'].mean():.2f} vs {df.loc[~has_ai, 'engagement_rate'].mean():.2f}.")

per_client_rate = df.groupby("client_id")["ai_traffic_pct"].apply(lambda s: (s > 0).mean())
print(f"3) It's very unevenly spread across clients: per-client share of pages with AI traffic ranges from "
      f"{per_client_rate.min()*100:.1f}% to {per_client_rate.max()*100:.1f}% (median {per_client_rate.median()*100:.1f}%) across {df['client_id'].nunique()} clients.")


1) Base rate: 6.08% of pages (568 of 9349) show any AI-assistant referral traffic.
2) Those pages already run hotter on organic signals: median clicks_90d is 20 vs 1 for the rest; mean engagement_rate is 3.43 vs 2.73.
3) It's very unevenly spread across clients: per-client share of pages with AI traffic ranges from 0.0% to 22.4% (median 2.8%) across 32 clients.


## 4. Careful words: what I can and can't claim


**What this work will be able to say:**
- *Observed:* in this anonymized sample, pages with any AI-assistant referral traffic have
  higher median organic clicks and higher mean engagement than pages without it.
- *Directional / decision-support:* a model trained on this data can rank pages by estimated
  likelihood of picking up AI-referral traffic, to help an editor decide where to spend
  restructuring effort first — not a guarantee of what will happen to any single page.

**What it will never say:**
- No causal claims — this is observational, cross-sectional data with no experiment, so I
  can't say restructuring a page *causes* it to gain AI-referral traffic, only that certain
  patterns are associated with pages that already have it.
- No "predicted [an AI provider's or Google's] algorithm" — the model only learns the pattern
  in already-observed, anonymized traffic outcomes, not the mechanics of any ranking or
  citation system.
- No generalizing past this sample — 32 clients, one anonymized snapshot; a pattern found
  here is a hypothesis to test on new data, not a settled fact.
- Flagging the label itself as noisy: `ai_traffic_pct` is zero-inflated (only 6.4% nonzero)
  with some outlier-looking values, so I'll treat it as a rough directional signal, not a
  precise measurement.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ done ] Every section above is filled — markdown thinking AND the code that backs it
- [ done ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [done  ] No client names, URLs, or private queries anywhere
- [ done ] My claims use careful words: observed, measured, directional, decision-support
- [done  ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.